# Train Drone Type Classification Model on Google Colab (FREE GPU)

This notebook trains a **Stage 2 classifier** to identify drone types:
- Cinewhoop
- DJI FPV
- DJI Mavic
- DJI Phantom
- Fixed wing
- Hexacopter
- Octocopter
- Pluto Mini Drone
- Quadcopter
- VTOL

**Setup Instructions:**
1. Upload this notebook to Google Colab
2. Go to Runtime → Change runtime type → Select **T4 GPU**
3. Run all cells

Training time: ~1-2 hours on GPU (vs 8+ hours on Mac CPU)

## 1. Install Dependencies

In [ ]:
!pip install torch torchvision roboflow pillow matplotlib tqdm -q
print("✓ Dependencies installed")

## 2. Download Dataset from Roboflow

In [ ]:
from roboflow import Roboflow
import yaml
from pathlib import Path

# Initialize Roboflow with your API key
API_KEY = "aO6VaU58uJ4WMpjdwVWU"  # Replace with your key if different
rf = Roboflow(api_key=API_KEY)

# Download drone classification dataset
print("Downloading dataset from Roboflow...")
project = rf.workspace("oleksandr-gorpynich").project("drone-detect-suvzw-gptrh")

# Try version 1 first, adjust if needed
try:
    dataset = project.version(1).download("yolov8")
except:
    # If version 1 doesn't exist, try version 2
    print("Version 1 not found, trying version 2...")
    dataset = project.version(2).download("yolov8")

dataset_path = Path(dataset.location)
print(f"\n✓ Dataset downloaded to: {dataset_path}")

# Read and display dataset info
with open(dataset_path / "data.yaml", 'r') as f:
    data_config = yaml.safe_load(f)

# Handle both list and dict formats for class names
names_data = data_config['names']
if isinstance(names_data, dict):
    class_names_list = list(names_data.values())
    class_names_dict = names_data
elif isinstance(names_data, list):
    class_names_list = names_data
    class_names_dict = {i: name for i, name in enumerate(names_data)}
else:
    raise ValueError(f"Unexpected format for 'names' in data.yaml: {type(names_data)}")

num_classes = len(class_names_list)

print(f"\nClasses ({num_classes}):")
for i, name in enumerate(class_names_list):
    print(f"  {i}: {name}")

## 3. Extract Drone Crops from Detection Dataset

Since this is a detection dataset (full images with bounding boxes), we need to crop out the drones first.

In [ ]:
import cv2
import shutil
from tqdm.auto import tqdm

def extract_crops_from_yolo_dataset(dataset_path, output_path, padding=10, min_size=32):
    """
    Extract crops from YOLO detection dataset for classification training.
    """
    dataset_path = Path(dataset_path)
    output_path = Path(output_path)
    
    # Read data.yaml
    with open(dataset_path / "data.yaml", 'r') as f:
        data_config = yaml.safe_load(f)
    
    # Handle both list and dict formats
    names_data = data_config['names']
    if isinstance(names_data, dict):
        class_names = names_data
    elif isinstance(names_data, list):
        class_names = {i: name for i, name in enumerate(names_data)}
    
    stats = {}
    
    # Process each split
    for split in ['train', 'valid', 'test']:
        split_path = dataset_path / split
        if not split_path.exists():
            continue
        
        # Create output directories for each class
        for class_id, class_name in class_names.items():
            class_dir = output_path / split / class_name
            class_dir.mkdir(parents=True, exist_ok=True)
        
        images_path = split_path / "images"
        labels_path = split_path / "labels"
        
        stats[split] = {name: 0 for name in class_names.values()}
        
        # Get all images
        image_files = list(images_path.glob("*.jpg")) + \
                     list(images_path.glob("*.jpeg")) + \
                     list(images_path.glob("*.png"))
        
        print(f"\nProcessing {split} split ({len(image_files)} images)...")
        
        for img_path in tqdm(image_files, desc=f"  Extracting {split} crops"):
            # Read image
            image = cv2.imread(str(img_path))
            if image is None:
                continue
            
            img_height, img_width = image.shape[:2]
            
            # Find corresponding label file
            label_path = labels_path / f"{img_path.stem}.txt"
            if not label_path.exists():
                continue
            
            # Read YOLO annotations
            with open(label_path, 'r') as f:
                lines = f.readlines()
            
            # Extract each detection as a crop
            for det_idx, line in enumerate(lines):
                parts = line.strip().split()
                if len(parts) < 5:
                    continue
                
                class_id = int(parts[0])
                x_center = float(parts[1])
                y_center = float(parts[2])
                width = float(parts[3])
                height = float(parts[4])
                
                # Convert to pixel coordinates
                x1 = int((x_center - width / 2) * img_width)
                y1 = int((y_center - height / 2) * img_height)
                x2 = int((x_center + width / 2) * img_width)
                y2 = int((y_center + height / 2) * img_height)
                
                # Add padding
                x1 = max(0, x1 - padding)
                y1 = max(0, y1 - padding)
                x2 = min(img_width, x2 + padding)
                y2 = min(img_height, y2 + padding)
                
                # Skip small crops
                if (x2 - x1) < min_size or (y2 - y1) < min_size:
                    continue
                
                # Extract crop
                crop = image[y1:y2, x1:x2]
                if crop.size == 0:
                    continue
                
                # Save crop
                class_name = class_names[class_id]
                crop_filename = f"{img_path.stem}_crop{det_idx}.jpg"
                crop_path = output_path / split / class_name / crop_filename
                
                cv2.imwrite(str(crop_path), crop)
                stats[split][class_name] += 1
    
    return stats

# Extract crops
classification_dataset = Path("drone_classification")
print("Extracting drone crops from detection dataset...")
stats = extract_crops_from_yolo_dataset(dataset_path, classification_dataset)

# Print statistics
print("\n" + "="*60)
print("DATASET STATISTICS")
print("="*60)
for split in ['train', 'valid', 'test']:
    if split in stats:
        print(f"\n{split.upper()}:")
        total = 0
        for class_name, count in sorted(stats[split].items()):
            print(f"  {class_name:20s}: {count:5d} images")
            total += count
        print(f"  {'Total':20s}: {total:5d} images")
print("="*60)

## 4. Prepare Data Loaders with Augmentation

In [ ]:
import torch
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder
from torch.utils.data import DataLoader

# Training transforms with aggressive augmentation
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomCrop((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
    transforms.RandomGrayscale(p=0.05),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Validation/Test transforms (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets
train_dataset = ImageFolder(classification_dataset / "train", transform=train_transform)
valid_dataset = ImageFolder(classification_dataset / "valid", transform=val_transform)
test_dataset = ImageFolder(classification_dataset / "test", transform=val_transform)

# Create data loaders
batch_size = 32
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
valid_loader = DataLoader(valid_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

# Get class names
class_names = train_dataset.classes
num_classes = len(class_names)

print(f"\nDataset loaded:")
print(f"  Train: {len(train_dataset)} images")
print(f"  Valid: {len(valid_dataset)} images")
print(f"  Test:  {len(test_dataset)} images")
print(f"  Classes: {num_classes}")
print(f"  Batch size: {batch_size}")

## 5. Create EfficientNet-B0 Model

In [ ]:
import torch.nn as nn
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load pretrained EfficientNet-B0
print("Loading pretrained EfficientNet-B0...")
model = efficientnet_b0(weights=EfficientNet_B0_Weights.IMAGENET1K_V1)

# Replace classifier head for our drone classes
in_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(in_features, num_classes)

model = model.to(device)

print(f"✓ Model ready with {num_classes} output classes")
print(f"  Input size: 224x224")
print(f"  Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

## 6. Training Configuration

In [ ]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

# Training hyperparameters
num_epochs = 40
learning_rate = 1e-3
weight_decay = 1e-4

# Calculate class weights for imbalanced dataset
class_counts = torch.zeros(num_classes)
for _, label in train_dataset:
    class_counts[label] += 1

class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum() * num_classes  # Normalize
class_weights = class_weights.to(device)

print(f"Class weights (for imbalance):")
for i, (name, weight) in enumerate(zip(class_names, class_weights)):
    print(f"  {name:20s}: {weight:.3f}")

# Loss function with class weights
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

# Learning rate scheduler
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs)

print(f"\nTraining configuration:")
print(f"  Epochs: {num_epochs}")
print(f"  Learning rate: {learning_rate}")
print(f"  Weight decay: {weight_decay}")
print(f"  Loss: CrossEntropyLoss with label smoothing (0.1)")
print(f"  Optimizer: AdamW")
print(f"  Scheduler: CosineAnnealingLR")

## 7. Training Loop (Fast on GPU!)

In [ ]:
from tqdm.auto import tqdm
import time

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc="Training")
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': running_loss / (pbar.n + 1), 'acc': 100. * correct / total})
    
    return running_loss / len(loader), 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        for images, labels in tqdm(loader, desc="Validation"):
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
    
    return running_loss / len(loader), 100. * correct / total

# Training loop
print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60 + "\n")

best_val_acc = 0.0
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

start_time = time.time()

for epoch in range(num_epochs):
    print(f"\nEpoch {epoch + 1}/{num_epochs}")
    print("-" * 60)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, device)
    
    # Validate
    val_loss, val_acc = validate(model, valid_loader, criterion, device)
    
    # Update learning rate
    scheduler.step()
    current_lr = optimizer.param_groups[0]['lr']
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print summary
    print(f"\nEpoch {epoch + 1} Summary:")
    print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%")
    print(f"  LR: {current_lr:.6f}")
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), 'best_classifier.pt')
        print(f"  ✓ New best model saved! (Val Acc: {val_acc:.2f}%)")

elapsed_time = time.time() - start_time
print(f"\n" + "="*60)
print(f"TRAINING COMPLETE!")
print(f"  Total time: {elapsed_time / 60:.1f} minutes")
print(f"  Best validation accuracy: {best_val_acc:.2f}%")
print("="*60)

## 8. Plot Training History

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(history['train_acc'], label='Train Acc')
ax2.plot(history['val_acc'], label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## 9. Evaluate on Test Set

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import seaborn as sns

# Load best model
model.load_state_dict(torch.load('best_classifier.pt'))
model.eval()

# Collect predictions
all_preds = []
all_labels = []

print("Evaluating on test set...")
with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        
        all_preds.extend(predicted.cpu().numpy())
        all_labels.extend(labels.numpy())

# Calculate metrics
test_acc = 100. * np.sum(np.array(all_preds) == np.array(all_labels)) / len(all_labels)

print(f"\n" + "="*60)
print(f"TEST SET RESULTS")
print("="*60)
print(f"\nOverall Accuracy: {test_acc:.2f}%\n")

# Per-class metrics
print("\nPer-Class Performance:")
print(classification_report(all_labels, all_preds, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=class_names, yticklabels=class_names)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 10. Visualize Predictions

In [ ]:
import random
from PIL import Image

# Get random test samples
num_samples = 12
indices = random.sample(range(len(test_dataset)), num_samples)

fig, axes = plt.subplots(3, 4, figsize=(16, 12))
axes = axes.flatten()

model.eval()
with torch.no_grad():
    for i, idx in enumerate(indices):
        # Get image and label
        img_tensor, true_label = test_dataset[idx]
        
        # Predict
        img_input = img_tensor.unsqueeze(0).to(device)
        output = model(img_input)
        probabilities = torch.softmax(output, dim=1)
        confidence, predicted = probabilities.max(1)
        
        predicted = predicted.item()
        confidence = confidence.item()
        
        # Denormalize image for display
        img = img_tensor.numpy().transpose(1, 2, 0)
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = std * img + mean
        img = np.clip(img, 0, 1)
        
        # Plot
        axes[i].imshow(img)
        axes[i].axis('off')
        
        true_name = class_names[true_label]
        pred_name = class_names[predicted]
        
        color = 'green' if predicted == true_label else 'red'
        title = f"True: {true_name}\nPred: {pred_name}\nConf: {confidence:.2%}"
        axes[i].set_title(title, color=color, fontsize=10)

plt.tight_layout()
plt.suptitle('Test Set Predictions', fontsize=16, y=1.02)
plt.show()

## 11. Download Trained Model

In [ ]:
from google.colab import files
import os

# Save complete model with class names
checkpoint = {
    'model_state_dict': model.state_dict(),
    'class_names': class_names,
    'num_classes': num_classes,
    'best_val_acc': best_val_acc,
    'test_acc': test_acc
}

torch.save(checkpoint, 'drone_classifier_complete.pt')

print("Downloading trained model...")
print(f"\nModel Info:")
print(f"  Validation Accuracy: {best_val_acc:.2f}%")
print(f"  Test Accuracy: {test_acc:.2f}%")
print(f"  Number of classes: {num_classes}")
print(f"  Classes: {', '.join(class_names)}")

files.download('drone_classifier_complete.pt')
print("\n✓ Download complete! Upload this to your Mac at:")
print("  models/checkpoints/drone_classifier/best.pt")

## 12. Test Single Image Inference

In [ ]:
def predict_single_image(model, image_path, transform, device, class_names):
    """
    Predict drone type for a single image.
    """
    from PIL import Image
    
    # Load and preprocess image
    image = Image.open(image_path).convert('RGB')
    image_tensor = transform(image).unsqueeze(0).to(device)
    
    # Predict
    model.eval()
    with torch.no_grad():
        output = model(image_tensor)
        probabilities = torch.softmax(output, dim=1)[0]
        confidence, predicted = probabilities.max(0)
    
    return class_names[predicted.item()], confidence.item(), probabilities.cpu().numpy()

# Test on a random image
test_image_path = random.choice([f for f in (classification_dataset / "test").rglob("*.jpg")])

pred_class, confidence, probs = predict_single_image(
    model, test_image_path, val_transform, device, class_names
)

print(f"\nSingle Image Prediction:")
print(f"  Image: {test_image_path.name}")
print(f"  Predicted: {pred_class}")
print(f"  Confidence: {confidence:.2%}")
print(f"\nTop 3 predictions:")
top3_indices = np.argsort(probs)[-3:][::-1]
for idx in top3_indices:
    print(f"  {class_names[idx]:20s}: {probs[idx]:.2%}")

# Display image
img = Image.open(test_image_path)
plt.figure(figsize=(8, 8))
plt.imshow(img)
plt.axis('off')
plt.title(f"Predicted: {pred_class} ({confidence:.2%})\nImage: {test_image_path.name}", fontsize=14)
plt.show()